# Volatility Forecasting — End-to-End Clean Notebook

## Goal

This notebook builds a self-contained realized volatility forecasting pipeline:

1. Load bucketed market data  
2. Construct lagged volatility and microstructure features  
3. Train walk-forward models with embargo  
4. Evaluate out-of-sample forecasts on 30m and 90m horizons  
5. Compare a baseline feature set against a richer v3 feature set  
6. Export the final 30m OOS prediction series for downstream stat-arb experiments

---

## Main questions

- Does the baseline walk-forward volatility model already have useful predictive power?
- Which feature additions provide the biggest improvement?
- Is it worth emphasizing high-volatility regimes in evaluation?
- Can the final 30m predicted volatility series be exported for downstream strategy overlay experiments?

In [1]:
import os, glob, math
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def expected_points_per_day(freq_min: float) -> int:
    return int(round(1440.0 / float(freq_min)))

def steps_for_minutes(window_min: int, freq_min: float) -> int:
    return max(1, int(round(window_min / float(freq_min))))

def coerce_timestamp_utc(s: pd.Series) -> pd.Series:
    ts = pd.to_datetime(s, errors="coerce")
    if getattr(ts.dt, "tz", None) is None:
        ts = ts.dt.tz_localize("UTC")
    else:
        ts = ts.dt.tz_convert("UTC")
    return ts

def freq_tag(freq_min: float) -> str:
    if abs(freq_min - round(freq_min)) < 1e-12:
        return str(int(round(freq_min)))
    s = f"{freq_min:.10f}".rstrip("0").rstrip(".")
    return s

def list_bucket_dates(buckets_root: str, freq_min: float):
    tag = freq_tag(freq_min)
    pat = os.path.join(buckets_root, f"freq={tag}min", "date=*", "buckets.parquet")
    files = sorted(glob.glob(pat))
    dates = [os.path.basename(os.path.dirname(f)).replace("date=", "") for f in files]
    return sorted(dates)

def filter_dates(dates, start=None, end=None, last_n=None):
    out = dates
    if start is not None:
        out = [d for d in out if d >= start]
    if end is not None:
        out = [d for d in out if d <= end]
    if last_n is not None and last_n > 0:
        out = out[-last_n:]
    return out

def load_buckets(buckets_root: str, freq_min: float, dates=None):
    tag = freq_tag(freq_min)
    if dates is None:
        pat = os.path.join(buckets_root, f"freq={tag}min", "date=*", "buckets.parquet")
        files = sorted(glob.glob(pat))
    else:
        files = []
        for d in dates:
            f = os.path.join(buckets_root, f"freq={tag}min", f"date={d}", "buckets.parquet")
            if os.path.exists(f):
                files.append(f)
        files = sorted(files)

    if not files:
        raise FileNotFoundError(
            f"No bucket parquet files found. Pattern tried: "
            f"{os.path.join(buckets_root, f'freq={tag}min', 'date=*', 'buckets.parquet')}"
        )

    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

    if "timestamp" not in df.columns:
        raise KeyError("Bucket parquet must contain 'timestamp' column.")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    if getattr(df["timestamp"].dt, "tz", None) is None:
        df["timestamp"] = df["timestamp"].dt.tz_localize("UTC")
    else:
        df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")

    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

def add_returns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "mid_last" in df.columns:
        px = df["mid_last"]
    elif "px_last" in df.columns:
        px = df["px_last"]
    else:
        raise KeyError("Need 'mid_last' or 'px_last' in buckets to compute returns.")
    px = px.astype(float).replace([np.inf, -np.inf], np.nan).ffill()
    df["ret"] = np.log(px).diff()
    df["ret2"] = df["ret"] ** 2
    return df

def add_features(df: pd.DataFrame, freq_min: float, har_windows_min=(5, 15, 60, 240)) -> pd.DataFrame:
    df = df.copy()
    for wmin in har_windows_min:
        w = steps_for_minutes(wmin, freq_min)
        df[f"rv_past_{wmin}m"] = df["ret2"].rolling(w, min_periods=w).sum()

    if "rel_spread_mean" in df.columns:
        w = steps_for_minutes(15, freq_min)
        df["spread_15m"] = df["rel_spread_mean"].rolling(w, min_periods=w).mean()

    if "imbalance_mean" in df.columns:
        w = steps_for_minutes(15, freq_min)
        df["imb_15m"] = df["imbalance_mean"].rolling(w, min_periods=w).mean()

    if "vol_notional" in df.columns:
        w = steps_for_minutes(15, freq_min)
        df["vol_15m"] = df["vol_notional"].rolling(w, min_periods=w).sum()

    return df

def add_labels(df: pd.DataFrame, freq_min: float, horizons_min=(30, 90), eps=1e-12) -> pd.DataFrame:
    df = df.copy()
    for Hmin in horizons_min:
        H = steps_for_minutes(Hmin, freq_min)
        rv_fwd = df["ret2"].rolling(H, min_periods=H).sum().shift(-H)
        df[f"y_{Hmin}m"] = np.log(rv_fwd + eps)
    return df

def infer_feat_cols(df: pd.DataFrame, y_col: str):
    prefixes = (
        "rv_past_",
        "absret_", "ret_std_",
        "spread_", "imb_", "vol_", "ntr_", "buy_",
        "jump_",
        "tod_",
    )
    drop = {"timestamp", "date", y_col}
    drop |= {c for c in df.columns if c.startswith("y_")}

    cols = []
    for c in df.columns:
        if c in drop:
            continue
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        if c.startswith(prefixes):
            cols.append(c)
    return sorted(cols)

def walk_forward_preds(
    df: pd.DataFrame,
    y_col: str,
    H_min: int,
    train_days: int,
    test_days: int,
    freq_min: float,
    ridge_alpha: float = 10.0,
    min_test_coverage: float = 0.90,
    feat_cols=None,
    use_weight=False,
    weight_lambda=2.0,
):
    df = df.sort_values("timestamp").reset_index(drop=True).copy()
    df["date"] = df["timestamp"].dt.floor("D")

    if feat_cols is None:
        feat_cols = infer_feat_cols(df, y_col)

    if (feat_cols is None) or (len(feat_cols) == 0):
        return pd.DataFrame(), pd.DataFrame()

    usable = df.dropna(subset=["timestamp", y_col] + feat_cols).copy()
    if usable.empty:
        return pd.DataFrame(), pd.DataFrame()

    unique_days = sorted(usable["date"].unique())
    if len(unique_days) < train_days + test_days + 2:
        return pd.DataFrame(), pd.DataFrame()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ridge_alpha))
    ])

    expected_test_unique = expected_points_per_day(freq_min) * test_days
    fold_rows = []
    pred_rows = []

    for i in range(train_days, len(unique_days) - test_days):
        train_end_day = unique_days[i]
        train_start_day = unique_days[i - train_days]
        train_mask = (usable["date"] >= train_start_day) & (usable["date"] < train_end_day)

        gap_start = pd.Timestamp(train_end_day)
        if gap_start.tz is None:
            gap_start = gap_start.tz_localize("UTC")
        gap_end = gap_start + pd.Timedelta(minutes=H_min)

        test_start = gap_end
        test_end = test_start + pd.Timedelta(days=test_days)
        test_mask = (usable["timestamp"] >= test_start) & (usable["timestamp"] < test_end)

        test_unique = usable.loc[test_mask, "timestamp"].nunique()
        if test_unique < min_test_coverage * expected_test_unique:
            continue

        train_df = usable.loc[train_mask, feat_cols + [y_col]]
        test_df = usable.loc[test_mask, ["timestamp"] + feat_cols + [y_col]]

        if len(train_df) < 1000 or len(test_df) < 200:
            continue

        X_train = train_df[feat_cols].to_numpy()
        y_train = train_df[y_col].to_numpy()
        X_test = test_df[feat_cols].to_numpy()
        y_test = test_df[y_col].to_numpy()

        if use_weight:
            yq = pd.Series(y_train).rank(pct=True).to_numpy()
            sample_weight = 1.0 + weight_lambda * yq
            model.fit(X_train, y_train, ridge__sample_weight=sample_weight)
        else:
            model.fit(X_train, y_train)

        pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, pred)
        rmse = math.sqrt(mean_squared_error(y_test, pred))
        corr = np.corrcoef(y_test, pred)[0, 1] if len(y_test) > 5 else np.nan

        fold_rows.append({
            "y_col": y_col,
            "H_min": H_min,
            "train_start_day": train_start_day,
            "train_end_day_exclusive": train_end_day,
            "test_start": test_start,
            "test_end_exclusive": test_end,
            "n_test_unique": int(test_unique),
            "mae": float(mae),
            "rmse": float(rmse),
            "corr": float(corr),
        })

        pred_rows.append(pd.DataFrame({
            "timestamp": test_df["timestamp"].values,
            "y_true": y_test,
            "y_pred": pred,
            "y_col": y_col,
            "H_min": H_min,
        }))

    folds = pd.DataFrame(fold_rows)
    preds = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
    return folds, preds

def summarize_folds(folds: pd.DataFrame) -> pd.DataFrame:
    if folds.empty:
        return pd.DataFrame()
    return folds.groupby(["H_min", "y_col"]).agg(
        mae_mean=("mae", "mean"),
        rmse_mean=("rmse", "mean"),
        corr_mean=("corr", "mean"),
        mae_std=("mae", "std"),
        rmse_std=("rmse", "std"),
        corr_std=("corr", "std"),
        folds=("mae", "count"),
    ).reset_index()

def make_combo(preds: pd.DataFrame, short_col="y_30m", long_col="y_90m", w=0.7):
    pS = preds[preds["y_col"] == short_col][["timestamp", "y_true", "y_pred"]].rename(
        columns={"y_true": "y_true_S", "y_pred": "y_pred_S"}
    )
    pL = preds[preds["y_col"] == long_col][["timestamp", "y_true", "y_pred"]].rename(
        columns={"y_true": "y_true_L", "y_pred": "y_pred_L"}
    )
    m = pS.merge(pL, on="timestamp", how="inner").sort_values("timestamp").reset_index(drop=True)

    rvS_hat = np.exp(m["y_pred_S"].to_numpy())
    rvL_hat = np.exp(m["y_pred_L"].to_numpy())
    m["y_pred_combo"] = np.log(w * rvS_hat + (1 - w) * rvL_hat + 1e-12)

    rvS_true = np.exp(m["y_true_S"].to_numpy())
    rvL_true = np.exp(m["y_true_L"].to_numpy())
    m["y_true_combo"] = np.log(w * rvS_true + (1 - w) * rvL_true + 1e-12)

    m["w_combo"] = w
    return m

def run_pipeline(
    buckets_root: str,
    out_dir: str,
    freq_min: float = 1.0,
    train_days: int = 30,
    test_days: int = 1,
    horizons=(30, 90),
    start: str = None,
    end: str = None,
    last_n: int = None,
    ridge_alpha: float = 10.0,
    min_test_coverage: float = 0.90,
    w_combo: float = 0.7,
    combo_short: int = 30,
    combo_long: int = 90,
):
    ensure_dir(out_dir)

    all_dates = list_bucket_dates(buckets_root, freq_min)
    if not all_dates:
        raise FileNotFoundError("No bucket dates found under buckets_root with this freq_min.")
    dates = filter_dates(all_dates, start=start, end=end, last_n=last_n)

    print(f"[INFO] Found {len(all_dates)} days, using {len(dates)} days: {dates[0]} .. {dates[-1]}")

    df = load_buckets(buckets_root, freq_min, dates=dates)
    df = add_returns(df)
    df = add_features(df, freq_min=freq_min)
    df = add_labels(df, freq_min=freq_min, horizons_min=tuple(horizons))

    all_folds, all_preds = [], []
    for H in horizons:
        y_col = f"y_{H}m"
        folds, preds = walk_forward_preds(
            df, y_col=y_col, H_min=H,
            train_days=train_days, test_days=test_days,
            freq_min=freq_min,
            ridge_alpha=ridge_alpha,
            min_test_coverage=min_test_coverage
        )
        if folds.empty:
            print(f"[WARN] No folds for H={H}")
            continue
        all_folds.append(folds)
        all_preds.append(preds)
        print(f"[INFO] H={H}: folds={len(folds)} preds_rows={len(preds)}")

    folds = pd.concat(all_folds, ignore_index=True) if all_folds else pd.DataFrame()
    preds = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    summary = summarize_folds(folds)

    summary.to_parquet(os.path.join(out_dir, "summary.parquet"), index=False)
    folds.to_parquet(os.path.join(out_dir, "folds.parquet"), index=False)
    preds.to_parquet(os.path.join(out_dir, "preds.parquet"), index=False)

    combo = None
    short_col = f"y_{combo_short}m"
    long_col = f"y_{combo_long}m"
    if (not preds.empty) and (short_col in set(preds["y_col"])) and (long_col in set(preds["y_col"])):
        combo = make_combo(preds, short_col=short_col, long_col=long_col, w=w_combo)
        combo.to_parquet(os.path.join(out_dir, "combo.parquet"), index=False)

    return summary, folds, preds, combo

## Baseline walk-forward run

We first run a simple baseline volatility pipeline using a compact feature set:

- HAR-style lagged realized volatility
- optional basic microstructure aggregates if present
- ridge regression
- rolling walk-forward evaluation
- embargo equal to forecast horizon

This establishes the benchmark before introducing richer features.

In [2]:
summary, folds, preds, combo = run_pipeline(
    buckets_root="/Volumes/profit/feature_store/buckets",
    out_dir="/Volumes/profit/feature_store/wf_out",
    freq_min=1.0,
    train_days=30,
    test_days=1,
    horizons=(30, 90),
    start="20251001",
    end="20260310",
    w_combo=0.7,
    combo_short=30,
    combo_long=90,
)

summary

[INFO] Found 139 days, using 139 days: 20251022 .. 20260310
[INFO] H=30: folds=104 preds_rows=149680
[INFO] H=90: folds=104 preds_rows=149680


,H_min,y_col,mae_mean,rmse_mean,corr_mean,mae_std,rmse_std,corr_std,folds
0,30,y_30m,0.706434,0.905370,0.557162,0.266272,0.366338,0.151319,104
1,90,y_90m,0.673781,0.855705,0.489423,0.268767,0.327361,0.170842,104


In [3]:
combo is None, (None if combo is None else combo.shape)
combo.head() if combo is not None else None

,timestamp,y_true_S,y_pred_S,y_true_L,y_pred_L,y_pred_combo,y_true_combo,w_combo
0,2025-11-21 01:30:00,-10.909746,-11.109231,-8.893863,-9.949897,-10.604619,-9.827193,0.7
1,2025-11-21 01:31:00,-10.988605,-11.157879,-8.898589,-9.980712,-10.642931,-9.849007,0.7
2,2025-11-21 01:32:00,-10.989200,-11.167180,-8.904692,-9.991622,-10.653169,-9.853873,0.7
3,2025-11-21 01:33:00,-10.937235,-11.173243,-8.904363,-9.991639,-10.655712,-9.841695,0.7
4,2025-11-21 01:34:00,-10.939284,-11.192074,-8.893673,-10.006246,-10.672079,-9.833973,0.7


## Baseline evaluation

We evaluate the baseline on both log-volatility and realized-variance scales.

Main metrics:

- MAE / RMSE / correlation in log-vol space
- QLIKE on realized variance
- RV-relative error diagnostics
- Top-quantile evaluation for high-volatility periods

In [4]:
def eval_one(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return mae, rmse, corr

p30 = preds[preds["y_col"] == "y_30m"]
p90 = preds[preds["y_col"] == "y_90m"]

mae30, rmse30, c30 = eval_one(p30["y_true"], p30["y_pred"])
mae90, rmse90, c90 = eval_one(p90["y_true"], p90["y_pred"])

maec, rmsec, cc = eval_one(combo["y_true_combo"], combo["y_pred_combo"]) if combo is not None else (np.nan, np.nan, np.nan)

pd.DataFrame({
    "series": ["30m", "90m", "combo"],
    "mae_log": [mae30, mae90, maec],
    "rmse_log": [rmse30, rmse90, rmsec],
    "corr_log": [c30, c90, cc],
})

,series,mae_log,rmse_log,corr_log
0,30m,0.706594,0.976255,0.644127
1,90m,0.673927,0.915853,0.636082
2,combo,0.665911,0.917356,0.643886


In [5]:
def _safe_exp(x):
    x = np.clip(np.asarray(x), -50, 50)
    return np.exp(x)

def qlike(rv_true, rv_pred, eps=1e-12):
    rv_true = np.asarray(rv_true)
    rv_pred = np.asarray(rv_pred)
    rv_pred = np.maximum(rv_pred, eps)
    ratio = np.maximum(rv_true, eps) / rv_pred
    return ratio - np.log(ratio) - 1.0

def eval_log_and_rv(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae_log = mean_absolute_error(y_true, y_pred)
    rmse_log = math.sqrt(mean_squared_error(y_true, y_pred))
    corr_log = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 5 else np.nan

    rv_true = _safe_exp(y_true)
    rv_pred = _safe_exp(y_pred)

    mae_rv = mean_absolute_error(rv_true, rv_pred)
    rmse_rv = math.sqrt(mean_squared_error(rv_true, rv_pred))
    qlike_mean = float(np.mean(qlike(rv_true, rv_pred)))

    return {
        "mae_log": float(mae_log),
        "rmse_log": float(rmse_log),
        "corr_log": float(corr_log),
        "mae_rv": float(mae_rv),
        "rmse_rv": float(rmse_rv),
        "qlike_mean": qlike_mean,
    }

def eval_preds_table(preds):
    rows = []
    for (y_col, H), g in preds.groupby(["y_col", "H_min"]):
        m = eval_log_and_rv(g["y_true"], g["y_pred"])
        rows.append({
            "y_col": y_col,
            "H_min": H,
            "n": len(g),
            **m
        })
    return pd.DataFrame(rows).sort_values("H_min")

def eval_combo_table(combo):
    if combo is None or combo.empty:
        return None
    m = eval_log_and_rv(combo["y_true_combo"], combo["y_pred_combo"])
    return pd.DataFrame([{
        "y_col": "combo",
        "H_min": -1,
        "n": len(combo),
        **m
    }])

baseline_metrics = eval_preds_table(preds)
baseline_combo_metrics = eval_combo_table(combo) if combo is not None else None

baseline_metrics

,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,149680,0.706594,0.976255,0.644127,1267.170416,254010.844183,0.558643
1,y_90m,90,149680,0.673927,0.915853,0.636082,19.238943,3303.905312,0.534888


In [6]:
baseline_combo_metrics

,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,combo,-1,149514,0.665911,0.917356,0.643886,893.782207,178877.400832,0.507174


In [7]:
def rv_relative_metrics(y_true, y_pred, eps=1e-12):
    rv_true = np.exp(np.clip(np.asarray(y_true), -50, 50))
    rv_pred = np.exp(np.clip(np.asarray(y_pred), -50, 50))
    rv_true = np.maximum(rv_true, eps)
    rv_pred = np.maximum(rv_pred, eps)
    rel = (rv_pred - rv_true) / rv_true
    return {
        "mape_rv": float(np.mean(np.abs(rel))),
        "rmspe_rv": float(np.sqrt(np.mean(rel**2))),
    }

rows = []
for (y_col, H), g in preds.groupby(["y_col", "H_min"]):
    rows.append({
        "y_col": y_col,
        "H_min": H,
        **rv_relative_metrics(g["y_true"], g["y_pred"])
    })

pd.DataFrame(rows).sort_values("H_min")

,y_col,H_min,mape_rv,rmspe_rv
0,y_30m,30,1.706429e+07,3.392323e+09
1,y_90m,90,1.153207e+05,2.041025e+07


In [8]:
def eval_top_quantile(preds, q=0.9):
    out = []
    for (y_col, H), g in preds.groupby(["y_col", "H_min"]):
        thr = np.quantile(g["y_true"], q)
        gg = g[g["y_true"] >= thr]
        m = eval_log_and_rv(gg["y_true"], gg["y_pred"])
        out.append({
            "y_col": y_col,
            "H_min": H,
            "q": q,
            "n": len(gg),
            **m
        })
    return pd.DataFrame(out).sort_values("H_min")

eval_top_quantile(preds, q=0.9)

,y_col,H_min,q,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,0.9,14968,1.384991,1.808173,0.217812,12671.704075,803252.817999,2.597794
1,y_90m,90,0.9,14968,1.348004,1.682528,0.197383,192.389231,10447.865960,2.551020


## Raw data sanity check

Before building richer features, we inspect the raw return series to confirm:

- date filtering is correct
- returns are computed as expected
- no obvious loading issue exists
- the empirical scale of returns is sensible

In [9]:
buckets_root = "/Volumes/profit/feature_store/buckets"
freq_min = 1.0
start = "20251001"
end = "20260310"

all_dates = list_bucket_dates(buckets_root, freq_min)
dates = filter_dates(all_dates, start=start, end=end, last_n=None)

df = load_buckets(buckets_root, freq_min, dates=dates)
df = add_returns(df)

df[["timestamp", "ret", "ret2"]].head()

,timestamp,ret,ret2
0,2025-10-22 15:16:00+00:00,NaN,NaN
1,2025-10-22 15:17:00+00:00,0.000060,3.576221e-09
2,2025-10-22 15:18:00+00:00,0.000573,3.286128e-07
3,2025-10-22 15:19:00+00:00,0.000110,1.218372e-08
4,2025-10-22 15:20:00+00:00,-0.000497,2.475047e-07


In [10]:
tmp = df[["ret"]].dropna()
tmp["ret"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99])

count    198046.000000
mean         -0.000002
std           0.000717
min          -0.054469
1%           -0.002067
10%          -0.000703
50%           0.000000
90%           0.000686
99%           0.002069
max           0.013939
Name: ret, dtype: float64

## Richer feature set (v3)

The baseline model is intentionally compact.  
We now extend it with a richer v3 feature set including:

- HAR-style lagged realized volatility
- jump and tail proxies
- richer microstructure dynamics
- simple intraday seasonality terms

This lets us test whether the main improvement comes from feature engineering rather than changing the model class.

In [11]:
def add_features_v3(df: pd.DataFrame, freq_min: float):
    df = df.copy()
    assert "ret" in df.columns and "ret2" in df.columns

    def wsteps(mins):
        return max(1, int(round(mins / float(freq_min))))

    df["abs_ret"] = df["ret"].abs()

    # 1) HAR: past RV
    for wmin in (5, 15, 60, 240):
        w = wsteps(wmin)
        df[f"rv_past_{wmin}m"] = df["ret2"].rolling(w, min_periods=w).sum()

    # 2) Jump / tail proxies
    for wmin in (5, 15, 60):
        w = wsteps(wmin)
        df[f"absret_sum_{wmin}m"] = df["abs_ret"].rolling(w, min_periods=w).sum()
        df[f"absret_max_{wmin}m"] = df["abs_ret"].rolling(w, min_periods=w).max()
        df[f"ret_std_{wmin}m"] = df["ret"].rolling(w, min_periods=w).std()

    # 3) Microstructure level + dynamics
    if "rel_spread_mean" in df.columns:
        for wmin in (5, 15, 60):
            w = wsteps(wmin)
            s = df["rel_spread_mean"]
            df[f"spread_mean_{wmin}m"] = s.rolling(w, min_periods=w).mean()
            df[f"spread_std_{wmin}m"] = s.rolling(w, min_periods=w).std()
            df[f"spread_max_{wmin}m"] = s.rolling(w, min_periods=w).max()

    if "imbalance_mean" in df.columns:
        for wmin in (5, 15, 60):
            w = wsteps(wmin)
            x = df["imbalance_mean"]
            df[f"imb_mean_{wmin}m"] = x.rolling(w, min_periods=w).mean()
            df[f"imb_std_{wmin}m"] = x.rolling(w, min_periods=w).std()
            df[f"imb_range_{wmin}m"] = (
                x.rolling(w, min_periods=w).max() - x.rolling(w, min_periods=w).min()
            )

    if "vol_notional" in df.columns:
        for wmin in (5, 15, 60):
            w = wsteps(wmin)
            v = df["vol_notional"]
            df[f"vol_sum_{wmin}m"] = v.rolling(w, min_periods=w).sum()
            df[f"vol_std_{wmin}m"] = v.rolling(w, min_periods=w).std()
            df[f"vol_max_{wmin}m"] = v.rolling(w, min_periods=w).max()

    if "n_trades" in df.columns:
        for wmin in (5, 15, 60):
            w = wsteps(wmin)
            n = df["n_trades"]
            df[f"ntr_sum_{wmin}m"] = n.rolling(w, min_periods=w).sum()
            df[f"ntr_std_{wmin}m"] = n.rolling(w, min_periods=w).std()
            df[f"ntr_max_{wmin}m"] = n.rolling(w, min_periods=w).max()

    if "buy_ratio" in df.columns:
        for wmin in (5, 15, 60):
            w = wsteps(wmin)
            b = df["buy_ratio"]
            df[f"buy_mean_{wmin}m"] = b.rolling(w, min_periods=w).mean()
            df[f"buy_std_{wmin}m"] = b.rolling(w, min_periods=w).std()

    # 4) Intraday seasonality
    if "timestamp" in df.columns:
        minute_of_day = (df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute).astype(float)
        df["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
        df["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440.0)

    return df


def run_pipeline_v3(
    buckets_root: str,
    out_dir: str,
    freq_min: float = 1.0,
    train_days: int = 30,
    test_days: int = 1,
    horizons=(30, 90),
    start: str = None,
    end: str = None,
    last_n: int = None,
    ridge_alpha: float = 10.0,
    min_test_coverage: float = 0.90,
    w_combo: float = 0.7,
    combo_short: int = 30,
    combo_long: int = 90,
):
    ensure_dir(out_dir)

    all_dates = list_bucket_dates(buckets_root, freq_min)
    dates = filter_dates(all_dates, start=start, end=end, last_n=last_n)
    print(f"[INFO] Found {len(all_dates)} days, using {len(dates)} days: {dates[0]} .. {dates[-1]}")

    df = load_buckets(buckets_root, freq_min, dates=dates)
    df = add_returns(df)
    df = add_features_v3(df, freq_min=freq_min)
    df = add_labels(df, freq_min=freq_min, horizons_min=tuple(horizons))

    all_folds, all_preds = [], []
    for H in horizons:
        y_col = f"y_{H}m"
        folds_h, preds_h = walk_forward_preds(
            df, y_col=y_col, H_min=H,
            train_days=train_days, test_days=test_days,
            freq_min=freq_min,
            ridge_alpha=ridge_alpha,
            min_test_coverage=min_test_coverage
        )
        if not folds_h.empty:
            all_folds.append(folds_h)
            all_preds.append(preds_h)

    folds = pd.concat(all_folds, ignore_index=True) if all_folds else pd.DataFrame()
    preds = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    summary = summarize_folds(folds)

    combo_df = None
    short_col = f"y_{combo_short}m"
    long_col = f"y_{combo_long}m"
    if (not preds.empty) and (short_col in set(preds["y_col"])) and (long_col in set(preds["y_col"])):
        combo_df = make_combo(preds, short_col=short_col, long_col=long_col, w=w_combo)

    ensure_dir(out_dir)
    summary.to_parquet(os.path.join(out_dir, "summary.parquet"), index=False)
    folds.to_parquet(os.path.join(out_dir, "folds.parquet"), index=False)
    preds.to_parquet(os.path.join(out_dir, "preds.parquet"), index=False)
    if combo_df is not None:
        combo_df.to_parquet(os.path.join(out_dir, "combo.parquet"), index=False)

    return summary, folds, preds, combo_df

In [12]:
summary3, folds3, preds3, combo3 = run_pipeline_v3(
    buckets_root="/Volumes/profit/feature_store/buckets",
    out_dir="/Volumes/profit/feature_store/wf_out_v3",
    freq_min=1.0,
    train_days=30,
    test_days=1,
    horizons=(30, 90),
    start="20251001",
    end="20260310",
    w_combo=0.7,
    combo_short=30,
    combo_long=90,
)

print("=== FULL (baseline) ===")
display(eval_preds_table(preds))

print("=== FULL (v3) ===")
display(eval_preds_table(preds3))

print("=== TOP10% (baseline) ===")
display(eval_top_quantile(preds, q=0.9))

print("=== TOP10% (v3) ===")
display(eval_top_quantile(preds3, q=0.9))

[INFO] Found 139 days, using 139 days: 20251022 .. 20260310
=== FULL (baseline) ===


,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,149680,0.706594,0.976255,0.644127,1267.170416,254010.844183,0.558643
1,y_90m,90,149680,0.673927,0.915853,0.636082,19.238943,3303.905312,0.534888


=== FULL (v3) ===


,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,149275,0.516827,0.689940,0.838825,0.000010,0.000793,0.325584
1,y_90m,90,149275,0.513553,0.683817,0.816470,0.000027,0.001448,0.392627


=== TOP10% (baseline) ===


,y_col,H_min,q,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,0.9,14968,1.384991,1.808173,0.217812,12671.704075,803252.817999,2.597794
1,y_90m,90,0.9,14968,1.348004,1.682528,0.197383,192.389231,10447.865960,2.551020


=== TOP10% (v3) ===


,y_col,H_min,q,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,0.9,14928,0.812286,1.068462,0.263255,0.000064,0.002506,1.276665
1,y_90m,90,0.9,14928,0.890368,1.149411,0.133170,0.000156,0.004577,2.028703


## Export final 30m OOS prediction series

We export the 30m OOS prediction stream from the richer v3 model for use in downstream stat-arb overlay experiments.

In [13]:
pred_df = (
    preds3.loc[preds3["H_min"] == 30, ["timestamp", "y_true", "y_pred"]]
          .rename(columns={
              "y_true": "y_30m",
              "y_pred": "pred_y_30m",
          })
          .copy()
)

pred_df["timestamp"] = pd.to_datetime(pred_df["timestamp"], utc=True)
pred_df = (
    pred_df.sort_values("timestamp")
           .drop_duplicates(subset=["timestamp"], keep="last")
           .reset_index(drop=True)
)

print(pred_df.head())
print(pred_df.columns.tolist())
print(pred_df.shape)
print(pred_df["timestamp"].min(), pred_df["timestamp"].max())

pred_df.to_parquet("/Volumes/profit/stat_arb_test/spot_vol_oos_pred_30m.parquet", index=False)

                  timestamp      y_30m  pred_y_30m
0 2025-11-21 00:30:00+00:00 -10.819157  -10.564773
1 2025-11-21 00:31:00+00:00 -10.822318  -10.517735
2 2025-11-21 00:32:00+00:00 -11.023323  -10.397844
3 2025-11-21 00:33:00+00:00 -11.005065  -10.371121
4 2025-11-21 00:34:00+00:00 -10.938123  -10.316741
['timestamp', 'y_30m', 'pred_y_30m']
(149178, 3)
2025-11-21 00:30:00+00:00 2026-03-10 00:29:00+00:00


## Feature ablation study

To understand where the improvement comes from, we compare several feature-set variants:

- baseline HAR-style setup
- microstructure dynamics
- tail quantile features
- jump count features
- combinations of the above

This helps isolate which feature families are genuinely useful.

In [14]:
def add_features_toggle(
    df: pd.DataFrame,
    freq_min: float,
    use_tail_quantile=False,
    use_jump_count=False,
    use_micro_dynamics=False,
    use_intraday=False,
):
    df = df.copy()
    assert "ret" in df.columns and "ret2" in df.columns

    def wsteps(mins):
        return max(1, int(round(mins / float(freq_min))))

    df["abs_ret"] = df["ret"].abs()

    # baseline HAR
    for wmin in (5, 15, 60, 240):
        w = wsteps(wmin)
        df[f"rv_past_{wmin}m"] = df["ret2"].rolling(w, min_periods=w).sum()

    # baseline jump-lite
    for wmin in (5, 15, 60):
        w = wsteps(wmin)
        df[f"absret_sum_{wmin}m"] = df["abs_ret"].rolling(w, min_periods=w).sum()
        df[f"absret_max_{wmin}m"] = df["abs_ret"].rolling(w, min_periods=w).max()
        df[f"ret_std_{wmin}m"] = df["ret"].rolling(w, min_periods=w).std()

    if use_tail_quantile:
        w = wsteps(60)
        df["absret_q90_60m"] = df["abs_ret"].rolling(w, min_periods=w).quantile(0.90)
        df["absret_q99_60m"] = df["abs_ret"].rolling(w, min_periods=w).quantile(0.99)

    if use_jump_count:
        w = wsteps(60)
        ret_std60 = df["ret"].rolling(w, min_periods=w).std()
        jump3 = (df["abs_ret"] > 3.0 * ret_std60).astype(float)
        df["jump_cnt3_60m"] = jump3.rolling(w, min_periods=w).sum()

    if use_micro_dynamics:
        if "rel_spread_mean" in df.columns:
            for wmin in (5, 15, 60):
                w = wsteps(wmin)
                s = df["rel_spread_mean"]
                df[f"spread_std_{wmin}m"] = s.rolling(w, min_periods=w).std()
                df[f"spread_max_{wmin}m"] = s.rolling(w, min_periods=w).max()

        if "imbalance_mean" in df.columns:
            for wmin in (5, 15, 60):
                w = wsteps(wmin)
                x = df["imbalance_mean"]
                df[f"imb_std_{wmin}m"] = x.rolling(w, min_periods=w).std()
                df[f"imb_range_{wmin}m"] = (
                    x.rolling(w, min_periods=w).max() - x.rolling(w, min_periods=w).min()
                )

        if "vol_notional" in df.columns:
            for wmin in (5, 15, 60):
                w = wsteps(wmin)
                v = df["vol_notional"]
                df[f"vol_std_{wmin}m"] = v.rolling(w, min_periods=w).std()
                df[f"vol_max_{wmin}m"] = v.rolling(w, min_periods=w).max()

        if "n_trades" in df.columns:
            for wmin in (5, 15, 60):
                w = wsteps(wmin)
                n = df["n_trades"]
                df[f"ntr_std_{wmin}m"] = n.rolling(w, min_periods=w).std()
                df[f"ntr_max_{wmin}m"] = n.rolling(w, min_periods=w).max()

    if use_intraday and "timestamp" in df.columns:
        minute_of_day = (df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute).astype(float)
        df["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
        df["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440.0)

    return df


def run_one_setting(
    df_raw,
    freq_min,
    H=30,
    train_days=30,
    test_days=1,
    ridge_alpha=10.0,
    min_test_coverage=0.90,
    **feat_flags,
):
    df = add_returns(df_raw.copy())
    df = add_features_toggle(df, freq_min=freq_min, **feat_flags)
    df = add_labels(df, freq_min=freq_min, horizons_min=(H,))
    y_col = f"y_{H}m"

    feat_cols = infer_feat_cols(df, y_col)
    folds_h, preds_h = walk_forward_preds(
        df,
        y_col=y_col,
        H_min=H,
        train_days=train_days,
        test_days=test_days,
        freq_min=freq_min,
        ridge_alpha=ridge_alpha,
        min_test_coverage=min_test_coverage,
        feat_cols=feat_cols,
    )

    full = eval_preds_table(preds_h).iloc[0].to_dict()
    top = eval_top_quantile(preds_h, q=0.9).iloc[0].to_dict()
    return full, top

In [15]:
settings = [
    ("baseline",   dict(use_tail_quantile=False, use_jump_count=False, use_micro_dynamics=False, use_intraday=False)),
    ("+micro_dyn", dict(use_tail_quantile=False, use_jump_count=False, use_micro_dynamics=True,  use_intraday=True)),
    ("+tail_q",    dict(use_tail_quantile=True,  use_jump_count=False, use_micro_dynamics=False, use_intraday=False)),
    ("+jump_cnt",  dict(use_tail_quantile=False, use_jump_count=True,  use_micro_dynamics=False, use_intraday=False)),
    ("micro+tail", dict(use_tail_quantile=True,  use_jump_count=False, use_micro_dynamics=True,  use_intraday=True)),
    ("micro+jump", dict(use_tail_quantile=False, use_jump_count=True,  use_micro_dynamics=True,  use_intraday=True)),
    ("all",        dict(use_tail_quantile=True,  use_jump_count=True,  use_micro_dynamics=True,  use_intraday=True)),
]

results = []
for name, flags in settings:
    full, top = run_one_setting(df, freq_min=1.0, H=30, **flags)
    results.append({
        "setting": name,
        "FULL_corr":  full["corr_log"],
        "FULL_rmse":  full["rmse_log"],
        "FULL_qlike": full["qlike_mean"],
        "TOP_corr":   top["corr_log"],
        "TOP_rmse":   top["rmse_log"],
        "TOP_qlike":  top["qlike_mean"],
        "n_full": full["n"],
        "n_top":  top["n"],
    })

pd.DataFrame(results).sort_values(["TOP_qlike", "FULL_qlike"])

,setting,FULL_corr,FULL_rmse,FULL_qlike,TOP_corr,TOP_rmse,TOP_qlike,n_full,n_top
1,+micro_dyn,0.839120,0.688776,0.325058,0.271210,1.062590,1.250203,149275,14928
4,micro+tail,0.838932,0.689141,0.326249,0.270333,1.065047,1.255273,149275,14928
5,micro+jump,0.839160,0.688694,0.325435,0.268979,1.067982,1.260776,149275,14928
6,all,0.839145,0.688725,0.325891,0.268045,1.068860,1.261290,149275,14928
0,baseline,0.821520,0.721652,0.373973,0.302996,1.167374,1.615203,149807,14981
2,+tail_q,0.822946,0.718960,0.372252,0.283263,1.168038,1.618018,149807,14981
3,+jump_cnt,0.823171,0.718640,0.371882,0.289665,1.171684,1.622445,149807,14981


## Weighted training experiment

Finally, we test whether placing larger sample weights on higher-volatility training observations helps:

- unweighted ridge
- weighted ridge with higher weight on larger target values

We compare both full-sample and top-decile performance.

In [16]:
def run_eval(df_raw, freq_min=1.0, H=30, flags=None, use_weight=False, weight_lambda=2.0):
    if flags is None:
        flags = dict(
            use_tail_quantile=False,
            use_jump_count=False,
            use_micro_dynamics=True,
            use_intraday=True,
        )

    df = add_returns(df_raw.copy())
    df = add_features_toggle(df, freq_min=freq_min, **flags)
    df = add_labels(df, freq_min=freq_min, horizons_min=(H,))
    y_col = f"y_{H}m"
    feat_cols = infer_feat_cols(df, y_col)

    folds, preds_local = walk_forward_preds(
        df, y_col=y_col, H_min=H,
        train_days=30, test_days=1,
        freq_min=freq_min,
        ridge_alpha=10.0,
        min_test_coverage=0.90,
        feat_cols=feat_cols,
        use_weight=use_weight,
        weight_lambda=weight_lambda,
    )

    full = eval_preds_table(preds_local)
    top = eval_top_quantile(preds_local, q=0.9)
    return full, top

# 30m
full_u, top_u = run_eval(df, H=30, use_weight=False)
full_w, top_w = run_eval(df, H=30, use_weight=True, weight_lambda=2.0)

print("=== 30m unweighted FULL ===")
display(full_u)
print("=== 30m weighted   FULL ===")
display(full_w)

print("=== 30m unweighted TOP10% ===")
display(top_u)
print("=== 30m weighted   TOP10% ===")
display(top_w)

=== 30m unweighted FULL ===


,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,149275,0.515994,0.688776,0.83912,0.000015,0.002497,0.325058


=== 30m weighted   FULL ===


,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,149275,0.532937,0.70356,0.838362,0.00002,0.004696,0.291969


=== 30m unweighted TOP10% ===


,y_col,H_min,q,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,0.9,14928,0.806579,1.06259,0.27121,0.00011,0.007897,1.250203


=== 30m weighted   TOP10% ===


,y_col,H_min,q,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_30m,30,0.9,14928,0.760769,1.004855,0.282983,0.000164,0.014851,1.055478


In [17]:
# 90m
full_u90, top_u90 = run_eval(df, H=90, use_weight=False)
full_w90, top_w90 = run_eval(df, H=90, use_weight=True, weight_lambda=2.0)

print("=== 90m unweighted FULL ===")
display(full_u90)
print("=== 90m weighted   FULL ===")
display(full_w90)

print("=== 90m unweighted TOP10% ===")
display(top_u90)
print("=== 90m weighted   TOP10% ===")
display(top_w90)

=== 90m unweighted FULL ===


,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_90m,90,149275,0.512541,0.682051,0.816788,0.00004,0.00636,0.344443


=== 90m weighted   FULL ===


,y_col,H_min,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_90m,90,149275,0.536792,0.698917,0.813864,0.000068,0.017184,0.309947


=== 90m unweighted TOP10% ===


,y_col,H_min,q,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_90m,90,0.9,14928,0.889695,1.141839,0.149015,0.000289,0.020113,1.515705


=== 90m weighted   TOP10% ===


,y_col,H_min,q,n,mae_log,rmse_log,corr_log,mae_rv,rmse_rv,qlike_mean
0,y_90m,90,0.9,14928,0.839119,1.078029,0.151402,0.000564,0.054341,1.288648


## Final takeaways

### Main conclusions

1. The baseline walk-forward volatility pipeline already has meaningful OOS predictive power.  
2. Richer feature engineering provides the main performance lift.  
3. Microstructure dynamics are especially valuable.  
4. High-volatility regime evaluation is important for downstream trading use.  
5. The exported 30m OOS prediction series can be used as an overlay signal in stat-arb experiments.

### Practical conclusion

The strongest direction is not necessarily using a more complex model class, but rather improving the feature representation of short-horizon market state.